# 07 — Retrieval Evaluation and Benchmarking

## Purpose

This notebook is the final analysis layer for the retrieval component of the NLP Knowledge Discovery Platform. It compares:

1. **BM25** lexical retrieval;
2. **Semantic Retrieval** based on Sentence Transformer embeddings;
3. **Knowledge Graph-Enhanced Retrieval** based on graph-aware reranking.

It covers two distinct evaluation settings:

- a reproducible **synthetic silver-standard evaluation** generated from the local corpus;
- an optional **manually labelled qualitative case study** over real arXiv papers.

The notebook does not reimplement retrieval logic. It reads the deterministic CSV outputs produced by the experiment modules and creates comparison tables, query-level analyses, and report-ready figures.

## Prerequisites

Run the complete workflow before executing this notebook:

```bash
conda activate nlp-kg
./scripts/run_full_validation.sh --no-dashboard
```

The automatic benchmark requires the six CSV files under `reports/tables/`. The manual section is enabled when the corresponding files under `reports/manual_test/` exist.

## 1. Imports and Project Setup

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# Locate the repository root independently of the notebook launch directory.
current = Path.cwd().resolve()
for candidate in [current, *current.parents]:
    if (candidate / "src").exists() and (candidate / "config.yaml").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Could not find the repository root containing src/ and config.yaml."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

TABLES_DIR = PROJECT_ROOT / "reports" / "tables"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
MANUAL_DIR = PROJECT_ROOT / "reports" / "manual_test"
PROCESSED_DOCUMENTS_PATH = (
    PROJECT_ROOT / "data" / "processed" / "processed_documents.csv"
)
MANUAL_QUERIES_PATH = (
    PROJECT_ROOT / "data" / "evaluation" / "manual_retrieval_queries.json"
)

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

print("Project root:", PROJECT_ROOT)
print("Tables:", TABLES_DIR)
print("Figures:", FIGURES_DIR)
print("Manual case directory:", MANUAL_DIR)

## 2. File Definitions and Validation

In [ ]:
AUTOMATIC_METRIC_PATHS = {
    "BM25": TABLES_DIR / "bm25_metrics.csv",
    "Semantic": TABLES_DIR / "semantic_metrics.csv",
    "KG-Enhanced": TABLES_DIR / "kg_enhanced_metrics.csv",
}

AUTOMATIC_RESULT_PATHS = {
    "BM25": TABLES_DIR / "bm25_results.csv",
    "Semantic": TABLES_DIR / "semantic_results.csv",
    "KG-Enhanced": TABLES_DIR / "kg_enhanced_results.csv",
}

required_paths = [
    *AUTOMATIC_METRIC_PATHS.values(),
    *AUTOMATIC_RESULT_PATHS.values(),
]
missing_paths = [path for path in required_paths if not path.is_file()]

if missing_paths:
    formatted = "\n".join(f"- {path}" for path in missing_paths)
    raise FileNotFoundError(
        "Required retrieval outputs are missing. Run "
        "./scripts/run_full_validation.sh --no-dashboard first.\n"
        f"Missing files:\n{formatted}"
    )

print("All required automatic benchmark files are available.")

## 3. Load and Validate Automatic Metrics

In [ ]:
REQUIRED_METRIC_COLUMNS = {
    "method",
    "k",
    "precision_at_k",
    "recall_at_k",
    "mrr",
    "num_queries",
}

metric_frames = []
for method_label, path in AUTOMATIC_METRIC_PATHS.items():
    frame = pd.read_csv(path)
    missing_columns = REQUIRED_METRIC_COLUMNS.difference(frame.columns)
    if missing_columns:
        raise ValueError(
            f"{path.name} is missing columns: {sorted(missing_columns)}"
        )

    frame = frame.copy()
    frame["method_label"] = method_label
    metric_frames.append(frame)

comparison_df = pd.concat(metric_frames, ignore_index=True)
comparison_df["k"] = pd.to_numeric(comparison_df["k"], errors="raise").astype(int)
comparison_df = comparison_df.sort_values(
    ["k", "method_label"],
    kind="stable",
).reset_index(drop=True)

display(
    comparison_df[
        [
            "method_label",
            "k",
            "precision_at_k",
            "recall_at_k",
            "mrr",
            "num_queries",
        ]
    ]
)

### Interpretation boundary

The automatic queries and relevance labels are derived from the local corpus. They are deterministic and useful for reproducible method comparison, but they are not independent human judgements. All results in this section must therefore be described as a **synthetic silver-standard evaluation**.

## 4. Save the Unified Automatic Benchmark Table

In [ ]:
automatic_benchmark_path = TABLES_DIR / "retrieval_benchmark_comparison.csv"
comparison_df.to_csv(automatic_benchmark_path, index=False)
print("Saved:", automatic_benchmark_path)

## 5. Precision@K Comparison

In [ ]:
precision_pivot = comparison_df.pivot(
    index="k",
    columns="method_label",
    values="precision_at_k",
).sort_index()

display(precision_pivot)

axis = precision_pivot.plot(kind="bar", figsize=(9, 5))
axis.set_title("Automatic Synthetic Silver-Standard Precision@K")
axis.set_xlabel("K")
axis.set_ylabel("Precision@K")
axis.set_ylim(bottom=0)
axis.legend(title="Method")
plt.tight_layout()

precision_figure_path = FIGURES_DIR / "automatic_precision_at_k.png"
plt.savefig(precision_figure_path, dpi=200, bbox_inches="tight")
plt.show()
print("Saved:", precision_figure_path)

## 6. Recall@K Comparison

In [ ]:
recall_pivot = comparison_df.pivot(
    index="k",
    columns="method_label",
    values="recall_at_k",
).sort_index()

display(recall_pivot)

axis = recall_pivot.plot(kind="bar", figsize=(9, 5))
axis.set_title("Automatic Synthetic Silver-Standard Recall@K")
axis.set_xlabel("K")
axis.set_ylabel("Recall@K")
axis.set_ylim(bottom=0)
axis.legend(title="Method")
plt.tight_layout()

recall_figure_path = FIGURES_DIR / "automatic_recall_at_k.png"
plt.savefig(recall_figure_path, dpi=200, bbox_inches="tight")
plt.show()
print("Saved:", recall_figure_path)

## 7. Mean Reciprocal Rank Comparison

In [ ]:
# MRR is calculated once over the full ranking set and is repeated for every K
# in the metric CSV. Keep one value per method for the comparison.
mrr_df = (
    comparison_df[["method_label", "mrr"]]
    .drop_duplicates(subset=["method_label"])
    .sort_values("method_label")
    .reset_index(drop=True)
)

display(mrr_df)

axis = mrr_df.plot(
    x="method_label",
    y="mrr",
    kind="bar",
    legend=False,
    figsize=(8, 5),
)
axis.set_title("Automatic Synthetic Silver-Standard MRR")
axis.set_xlabel("Method")
axis.set_ylabel("MRR")
axis.set_ylim(bottom=0)
plt.xticks(rotation=0)
plt.tight_layout()

mrr_figure_path = FIGURES_DIR / "automatic_mrr.png"
plt.savefig(mrr_figure_path, dpi=200, bbox_inches="tight")
plt.show()
print("Saved:", mrr_figure_path)

## 8. Metric Leaders by K

In [ ]:
leader_rows = []
for k_value, group in comparison_df.groupby("k", sort=True):
    for metric_name in ["precision_at_k", "recall_at_k"]:
        best_value = group[metric_name].max()
        leaders = sorted(
            group.loc[group[metric_name] == best_value, "method_label"].tolist()
        )
        leader_rows.append(
            {
                "k": int(k_value),
                "metric": metric_name,
                "best_value": float(best_value),
                "leaders": ", ".join(leaders),
            }
        )

best_mrr = mrr_df["mrr"].max()
leader_rows.append(
    {
        "k": np.nan,
        "metric": "mrr",
        "best_value": float(best_mrr),
        "leaders": ", ".join(
            sorted(mrr_df.loc[mrr_df["mrr"] == best_mrr, "method_label"])
        ),
    }
)

leaders_df = pd.DataFrame(leader_rows)
display(leaders_df)

## 9. Load and Validate Automatic Ranked Results

In [ ]:
REQUIRED_RESULT_COLUMNS = {
    "method",
    "query_id",
    "query",
    "rank",
    "doc_id",
    "score",
    "relevant",
}

result_frames = {}
for method_label, path in AUTOMATIC_RESULT_PATHS.items():
    frame = pd.read_csv(path, dtype={"query_id": "string", "doc_id": "string"})
    missing_columns = REQUIRED_RESULT_COLUMNS.difference(frame.columns)
    if missing_columns:
        raise ValueError(
            f"{path.name} is missing columns: {sorted(missing_columns)}"
        )

    frame = frame.copy()
    frame["rank"] = pd.to_numeric(frame["rank"], errors="raise").astype(int)
    frame["relevant"] = frame["relevant"].astype(str).str.lower().eq("true")
    frame["method_label"] = method_label
    result_frames[method_label] = frame.sort_values(
        ["query_id", "rank"],
        kind="stable",
    ).reset_index(drop=True)

result_inventory = pd.DataFrame(
    [
        {
            "method": method_label,
            "rows": len(frame),
            "queries": frame["query_id"].nunique(),
            "max_rank": int(frame["rank"].max()),
            "relevant_hits": int(frame["relevant"].sum()),
        }
        for method_label, frame in result_frames.items()
    ]
)

display(result_inventory)

## 10. Query-Level Ranking Inspection

In [ ]:
query_ids = sorted(
    set().union(*(set(frame["query_id"].dropna()) for frame in result_frames.values()))
)
if not query_ids:
    raise ValueError("No query IDs were found in the retrieval result files.")

# Set this variable to a specific query ID to inspect another query.
SELECTED_QUERY_ID = query_ids[0]
TOP_N = 10

print("Selected query ID:", SELECTED_QUERY_ID)

for method_label, frame in result_frames.items():
    selected = frame.loc[
        frame["query_id"] == SELECTED_QUERY_ID,
        ["query_id", "query", "rank", "doc_id", "score", "relevant"],
    ].head(TOP_N)

    print(f"\n{method_label}")
    display(selected)

## 11. Correct Per-Query Semantic vs KG Ranking Comparison

In [ ]:
def compare_query_rankings(
    semantic_results: pd.DataFrame,
    kg_results: pd.DataFrame,
    query_id: str,
    top_n: int = 10,
) -> pd.DataFrame:
    """Compare Semantic and KG-enhanced ranks for one query only."""

    semantic_query = (
        semantic_results.loc[semantic_results["query_id"] == query_id]
        .sort_values("rank")
        .head(top_n)
        [["doc_id", "rank", "score", "relevant"]]
        .rename(
            columns={
                "rank": "semantic_rank",
                "score": "semantic_score",
                "relevant": "semantic_relevant",
            }
        )
    )

    kg_query = (
        kg_results.loc[kg_results["query_id"] == query_id]
        .sort_values("rank")
        .head(top_n)
        [["doc_id", "rank", "score", "relevant"]]
        .rename(
            columns={
                "rank": "kg_rank",
                "score": "kg_score",
                "relevant": "kg_relevant",
            }
        )
    )

    comparison = semantic_query.merge(kg_query, on="doc_id", how="outer")
    comparison["relevant"] = (
        comparison[["semantic_relevant", "kg_relevant"]]
        .fillna(False)
        .any(axis=1)
    )
    comparison["rank_change_semantic_minus_kg"] = (
        comparison["semantic_rank"] - comparison["kg_rank"]
    )
    comparison["status"] = np.select(
        [
            comparison["semantic_rank"].isna(),
            comparison["kg_rank"].isna(),
            comparison["rank_change_semantic_minus_kg"] > 0,
            comparison["rank_change_semantic_minus_kg"] < 0,
        ],
        [
            "entered KG top-N",
            "left KG top-N",
            "moved up in KG",
            "moved down in KG",
        ],
        default="same rank",
    )

    return comparison.sort_values(
        ["kg_rank", "semantic_rank"],
        na_position="last",
        kind="stable",
    ).reset_index(drop=True)

selected_ranking_comparison = compare_query_rankings(
    result_frames["Semantic"],
    result_frames["KG-Enhanced"],
    SELECTED_QUERY_ID,
    top_n=TOP_N,
)

display(selected_ranking_comparison)

A positive `rank_change_semantic_minus_kg` means that the document moved upward after KG-enhanced reranking. A document marked `entered KG top-N` was absent from the displayed Semantic top-N but appears in the KG-enhanced top-N. This is a ranking observation; improvement is established only when the relevance labels and evaluation metrics also improve.

## 12. Aggregate Per-Query Ranking Change Summary

In [ ]:
def build_ranking_change_summary(
    semantic_results: pd.DataFrame,
    kg_results: pd.DataFrame,
    top_n: int,
) -> pd.DataFrame:
    rows = []
    all_query_ids = sorted(
        set(semantic_results["query_id"]).union(kg_results["query_id"])
    )

    for query_id in all_query_ids:
        semantic_query = (
            semantic_results.loc[semantic_results["query_id"] == query_id]
            .sort_values("rank")
            .head(top_n)
        )
        kg_query = (
            kg_results.loc[kg_results["query_id"] == query_id]
            .sort_values("rank")
            .head(top_n)
        )

        semantic_docs = set(semantic_query["doc_id"])
        kg_docs = set(kg_query["doc_id"])
        union = semantic_docs.union(kg_docs)
        overlap = semantic_docs.intersection(kg_docs)

        rows.append(
            {
                "query_id": query_id,
                "top_n": top_n,
                "semantic_relevant_hits": int(semantic_query["relevant"].sum()),
                "kg_relevant_hits": int(kg_query["relevant"].sum()),
                "relevant_hit_delta": int(
                    kg_query["relevant"].sum() - semantic_query["relevant"].sum()
                ),
                "shared_documents": len(overlap),
                "jaccard_overlap": len(overlap) / len(union) if union else 0.0,
                "entered_kg_top_n": len(kg_docs - semantic_docs),
                "left_kg_top_n": len(semantic_docs - kg_docs),
            }
        )

    return pd.DataFrame(rows)

max_evaluated_k = int(comparison_df["k"].max())
ranking_change_summary = build_ranking_change_summary(
    result_frames["Semantic"],
    result_frames["KG-Enhanced"],
    top_n=max_evaluated_k,
)

display(ranking_change_summary)

ranking_change_summary_path = (
    TABLES_DIR / "retrieval_ranking_change_summary.csv"
)
ranking_change_summary.to_csv(ranking_change_summary_path, index=False)
print("Saved:", ranking_change_summary_path)

## 13. KG Evidence Inspection

In [ ]:
kg_results = result_frames["KG-Enhanced"]
kg_diagnostic_columns = [
    column
    for column in [
        "query_id",
        "rank",
        "doc_id",
        "score",
        "base_score",
        "normalized_base_score",
        "graph_score",
        "normalized_graph_score",
        "graph_evidence",
        "relevant",
    ]
    if column in kg_results.columns
]

if len(kg_diagnostic_columns) > 6:
    display(
        kg_results.loc[
            kg_results["query_id"] == SELECTED_QUERY_ID,
            kg_diagnostic_columns,
        ].head(TOP_N)
    )
else:
    print(
        "No optional graph-diagnostic columns are available in the current "
        "KG-enhanced result file."
    )

## 14. Automatic Retrieval Summary

In [ ]:
automatic_summary_rows = []
for method_label, frame in result_frames.items():
    automatic_summary_rows.append(
        {
            "method": method_label,
            "retrieved_rows": len(frame),
            "num_queries": frame["query_id"].nunique(),
            "max_rank": int(frame["rank"].max()),
            "relevant_hits_in_saved_rankings": int(frame["relevant"].sum()),
            "mean_score": float(frame["score"].mean()),
        }
    )

automatic_summary_df = pd.DataFrame(automatic_summary_rows)
display(automatic_summary_df)

automatic_summary_path = TABLES_DIR / "retrieval_summary.csv"
automatic_summary_df.to_csv(automatic_summary_path, index=False)
print("Saved:", automatic_summary_path)

## 15. Manual Qualitative Case Study

This section is optional and runs only when `reports/manual_test/` contains all three metric and result files. It must be interpreted separately from the automatic benchmark because it uses human-selected relevance labels.

In [ ]:
MANUAL_METRIC_PATHS = {
    "BM25": MANUAL_DIR / "bm25_metrics.csv",
    "Semantic": MANUAL_DIR / "semantic_metrics.csv",
    "KG-Enhanced": MANUAL_DIR / "kg_enhanced_metrics.csv",
}
MANUAL_RESULT_PATHS = {
    "BM25": MANUAL_DIR / "bm25_results.csv",
    "Semantic": MANUAL_DIR / "semantic_results.csv",
    "KG-Enhanced": MANUAL_DIR / "kg_enhanced_results.csv",
}

manual_files_available = all(
    path.is_file()
    for path in [
        *MANUAL_METRIC_PATHS.values(),
        *MANUAL_RESULT_PATHS.values(),
    ]
)

print("Manual case available:", manual_files_available)

In [ ]:
manual_comparison_df = pd.DataFrame()
manual_result_frames = {}

if manual_files_available:
    manual_metric_frames = []
    for method_label, path in MANUAL_METRIC_PATHS.items():
        frame = pd.read_csv(path)
        missing_columns = REQUIRED_METRIC_COLUMNS.difference(frame.columns)
        if missing_columns:
            raise ValueError(
                f"{path.name} is missing columns: {sorted(missing_columns)}"
            )
        frame = frame.copy()
        frame["method_label"] = method_label
        manual_metric_frames.append(frame)

    manual_comparison_df = (
        pd.concat(manual_metric_frames, ignore_index=True)
        .sort_values(["k", "method_label"], kind="stable")
        .reset_index(drop=True)
    )

    display(
        manual_comparison_df[
            [
                "method_label",
                "k",
                "precision_at_k",
                "recall_at_k",
                "mrr",
                "num_queries",
            ]
        ]
    )

    manual_benchmark_path = MANUAL_DIR / "manual_retrieval_benchmark.csv"
    manual_comparison_df.to_csv(manual_benchmark_path, index=False)
    print("Saved:", manual_benchmark_path)
else:
    print(
        "Manual outputs are not available. Run the full validation script "
        "without --skip-manual to enable this section."
    )

## 16. Manual Case Metric Visualization

In [ ]:
if manual_files_available:
    manual_recall = manual_comparison_df.pivot(
        index="k",
        columns="method_label",
        values="recall_at_k",
    ).sort_index()

    axis = manual_recall.plot(kind="bar", figsize=(9, 5))
    axis.set_title("Manual Qualitative Case Study: Recall@K")
    axis.set_xlabel("K")
    axis.set_ylabel("Recall@K")
    axis.set_ylim(bottom=0)
    axis.legend(title="Method")
    plt.tight_layout()

    manual_figure_path = FIGURES_DIR / "manual_retrieval_recall_at_k.png"
    plt.savefig(manual_figure_path, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved:", manual_figure_path)
else:
    print("Manual metric plot skipped because the manual outputs are missing.")

## 17. Manual Case Query and Ranked Results

In [ ]:
manual_query_metadata = []
if MANUAL_QUERIES_PATH.is_file():
    manual_query_metadata = json.loads(
        MANUAL_QUERIES_PATH.read_text(encoding="utf-8")
    )

if manual_query_metadata:
    display(pd.DataFrame(manual_query_metadata))
else:
    print("No manual query metadata file is available.")

if manual_files_available:
    documents = pd.DataFrame()
    if PROCESSED_DOCUMENTS_PATH.is_file():
        documents = pd.read_csv(
            PROCESSED_DOCUMENTS_PATH,
            dtype={"doc_id": "string"},
        )
        available_document_columns = [
            column
            for column in ["doc_id", "title", "categories"]
            if column in documents.columns
        ]
        documents = documents[available_document_columns].drop_duplicates("doc_id")

    for method_label, path in MANUAL_RESULT_PATHS.items():
        frame = pd.read_csv(
            path,
            dtype={"query_id": "string", "doc_id": "string"},
        )
        frame["rank"] = pd.to_numeric(frame["rank"], errors="raise").astype(int)
        frame["relevant"] = frame["relevant"].astype(str).str.lower().eq("true")
        frame["method_label"] = method_label

        if not documents.empty:
            frame = frame.merge(documents, on="doc_id", how="left")

        manual_result_frames[method_label] = frame.sort_values(
            ["query_id", "rank"],
            kind="stable",
        ).reset_index(drop=True)

        visible_columns = [
            column
            for column in [
                "query_id",
                "query",
                "rank",
                "doc_id",
                "score",
                "relevant",
                "title",
                "categories",
            ]
            if column in frame.columns
        ]

        print(f"\n{method_label}")
        display(frame[visible_columns].head(10))
else:
    print("Manual ranked-result inspection skipped.")

## 18. Manual Semantic vs KG Ranking Change

In [ ]:
if manual_files_available:
    manual_query_ids = sorted(
        set(manual_result_frames["Semantic"]["query_id"]).intersection(
            manual_result_frames["KG-Enhanced"]["query_id"]
        )
    )
    if not manual_query_ids:
        raise ValueError("No shared manual query IDs were found.")

    MANUAL_QUERY_ID = manual_query_ids[0]
    manual_ranking_comparison = compare_query_rankings(
        manual_result_frames["Semantic"],
        manual_result_frames["KG-Enhanced"],
        MANUAL_QUERY_ID,
        top_n=10,
    )

    if PROCESSED_DOCUMENTS_PATH.is_file():
        title_lookup = pd.read_csv(
            PROCESSED_DOCUMENTS_PATH,
            dtype={"doc_id": "string"},
        )
        if "title" in title_lookup.columns:
            title_lookup = title_lookup[["doc_id", "title"]].drop_duplicates("doc_id")
            manual_ranking_comparison = manual_ranking_comparison.merge(
                title_lookup,
                on="doc_id",
                how="left",
            )

    display(manual_ranking_comparison)

    manual_ranking_path = MANUAL_DIR / "manual_ranking_comparison.csv"
    manual_ranking_comparison.to_csv(manual_ranking_path, index=False)
    print("Saved:", manual_ranking_path)
else:
    print("Manual ranking comparison skipped.")

## 19. Data-Grounded Interpretation

In [ ]:
print("AUTOMATIC SYNTHETIC SILVER-STANDARD EVALUATION")
for _, row in leaders_df.iterrows():
    if pd.isna(row["k"]):
        print(
            f"- Best {row['metric']}: {row['leaders']} "
            f"({row['best_value']:.4f})"
        )
    else:
        print(
            f"- Best {row['metric']} at K={int(row['k'])}: "
            f"{row['leaders']} ({row['best_value']:.4f})"
        )

print(
    "\nInterpret these results as a reproducible synthetic silver-standard "
    "benchmark, not as independent human relevance assessment."
)

if manual_files_available:
    print("\nMANUAL QUALITATIVE CASE STUDY")
    for k_value, group in manual_comparison_df.groupby("k", sort=True):
        semantic_row = group.loc[group["method_label"] == "Semantic"]
        kg_row = group.loc[group["method_label"] == "KG-Enhanced"]
        if not semantic_row.empty and not kg_row.empty:
            recall_delta = (
                float(kg_row.iloc[0]["recall_at_k"])
                - float(semantic_row.iloc[0]["recall_at_k"])
            )
            print(
                f"- Recall delta KG-Enhanced minus Semantic at K={int(k_value)}: "
                f"{recall_delta:+.4f}"
            )

    print(
        "This manual result is a one-query qualitative case study and must "
        "not be presented as statistically conclusive evidence."
    )

## 20. Discussion

### BM25

BM25 is the lexical baseline. It is fast, interpretable, and often strong when query terms overlap directly with document text. It does not model contextual similarity beyond lexical statistics.

### Semantic Retrieval

Semantic retrieval can recover conceptually similar documents even when exact wording differs. Its ranking depends on the embedding model and the representation of the paper text.

### Knowledge Graph-Enhanced Retrieval

KG-enhanced retrieval reranks a larger candidate set using graph evidence in addition to the base retrieval score. A changed ranking is not automatically an improvement. Improvement must be supported by relevance labels and metrics such as Recall@K, Precision@K, or MRR.

### Evaluation limitations

- The automatic benchmark is synthetic silver-standard evaluation.
- The manual experiment contains one labelled query and is qualitative.
- The evaluated subset is not the complete arXiv corpus.
- Results may depend on the source snapshot, subset construction, embedding model, and graph extraction quality.
- Stronger general claims would require a larger independently labelled query set and statistical analysis.

## 21. Conclusion

This notebook provides a reproducible and transparent comparison of BM25, Semantic Retrieval, and Knowledge Graph-Enhanced Retrieval. It separates automatic silver-standard findings from the manual qualitative case study, performs ranking analysis per query, and avoids treating ranking changes alone as proof of improvement.

The correct project-level conclusion is conditional: graph-aware reranking can improve semantic retrieval for some queries by promoting graph-supported relevant documents, while broader claims require a larger human-labelled evaluation set.